In [ ]:
!pip install -q -U "transformers==5.14.1" datasets bitsandbytes accelerate statsmodels
from google.colab import drive
drive.mount("/content/drive")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 120.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 42.3 MB/s eta 0:00:00
Mounted at /content/drive


In [ ]:
import gc, json, os, random
import numpy as np, pandas as pd, torch
from datasets import load_dataset
from sklearn.metrics import f1_score
from statsmodels.stats.multitest import multipletests

PROJECT_ROOT = "/content/drive/MyDrive/EmotionDetection"
VALID_CKPT   = f"{PROJECT_ROOT}/CAST_checkpoints"

P2_DIR = f"{VALID_CKPT}/Phase2"
P3_DIR = f"{VALID_CKPT}/Phase3"
P4_DIR = f"{VALID_CKPT}/Phase4"
TEST_PRED_DIR    = f"{P4_DIR}/predictions"
SELECTION_PATH   = f"{P4_DIR}/phase4_serengeti_validation_selections.json"
RESULTS_PATH     = f"{P4_DIR}/phase4_serengeti_test_ablation_results.json"
FEWSHOT_PATH     = f"{P4_DIR}/phase4_serengeti_fewshot_only_validation_results.json"

P2_RESULTS      = f"{P2_DIR}/phase2_validation_raw_results_gemma4.json"
P3_SRNG_RESULTS = f"{P3_DIR}/phase3_serengeti_validation.json"
CACHE_DIR       = f"{P3_DIR}/data_cache"
MODEL_CACHE_DIR = f"{PROJECT_ROOT}/model_cache"
LOCAL_MODEL_CACHE = "/content/hf_cache_local"

for path in [P4_DIR, TEST_PRED_DIR, MODEL_CACHE_DIR, LOCAL_MODEL_CACHE]:
    os.makedirs(path, exist_ok=True)
os.environ.update(HF_HOME=MODEL_CACHE_DIR,
                  HF_DATASETS_CACHE=f"{MODEL_CACHE_DIR}/datasets",
                  HF_HUB_DISABLE_SYMLINKS_WARNING="1")

GEMMA_MODEL   = "unsloth/gemma-4-31B-it-unsloth-bnb-4bit"
BATCH_SIZE    = 8
FEW_SHOT_K    = 2
RANDOM_SEED   = 42
N_BOOT        = 10_000
GENERATION_CONFIG = {"max_new_tokens": 30, "do_sample": False}

LANG_ORDER   = ["eng","hin","rus","hau","kin","sun","yor","vmw","pcm"]
RUN_LANGS    = ["hau","kin","yor","vmw","pcm"]   # SERENGETI-supported only
PROMPT_ORDER = ["p0","p1","p2","p3"]
CONFIG_ORDER = ["C1","C2","C3"]
EMOTION_ORDER = ["anger","disgust","fear","joy","sadness","surprise"]
ABSENT_EMOTIONS = {"eng": {"disgust"}}

LANGUAGES = dict(zip(LANG_ORDER,[
    "English","Hindi","Russian","Hausa","Kinyarwanda","Sundanese",
    "Yoruba","Emakhuwa","Nigerian Pidgin"]))
LANGUAGES = {k: {"name": v} for k, v in LANGUAGES.items()}

ALL_TARGET_CODES = LANG_ORDER.copy()

# SERENGETI-specific pool construction
EXTRA_LANGS = {
    "deu": {"family":"Indo-European","genus":"Germanic"},
    "swe": {"family":"Indo-European","genus":"Germanic"},
    "afr": {"family":"Indo-European","genus":"Germanic"},
    "swa": {"family":"Niger-Congo","genus":"Bantu"},
    "ibo": {"family":"Niger-Congo","genus":"Volta-Niger"},
}
PCM_C2 = ["eng","deu","swe","afr"]
PCM_C3 = ["eng"]

FAMILY = {**{k: v.get("family","") for k, v in {
    "eng": {"family":"Indo-European"}, "hin": {"family":"Indo-European"},
    "rus": {"family":"Indo-European"}, "hau": {"family":"Afroasiatic"},
    "kin": {"family":"Niger-Congo"},   "sun": {"family":"Austronesian"},
    "yor": {"family":"Niger-Congo"},   "vmw": {"family":"Niger-Congo"},
    "pcm": {"family":"Creole"},
}.items()}, **{k: v["family"] for k, v in EXTRA_LANGS.items()}}
GENUS = {**{k: v.get("genus","") for k, v in {
    "eng": {"genus":"Germanic"},  "hin": {"genus":"Indo-Aryan"},
    "rus": {"genus":"Slavic"},    "hau": {"genus":"Chadic"},
    "kin": {"genus":"Bantu"},     "sun": {"genus":"Sundic"},
    "yor": {"genus":"Volta-Niger"},"vmw": {"genus":"Bantu"},
    "pcm": {"genus":"English-Lexifier"},
}.items()}, **{k: v["genus"] for k, v in EXTRA_LANGS.items()}}

def get_c1_pool(target): return [c for c in ALL_TARGET_CODES if c != target]
def get_c2_pool(target):
    if target == "pcm": return PCM_C2.copy()
    f = FAMILY[target]
    return [c for c in FAMILY if c != target and FAMILY[c] == f]
def get_c3_pool(target):
    if target == "pcm": return PCM_C3.copy()
    if target in ("hau","sun"): return get_c2_pool(target)
    g = GENUS[target]
    return [c for c in GENUS if c != target and GENUS[c] == g]

POOL_FUNCTIONS = {"C1": get_c1_pool, "C2": get_c2_pool, "C3": get_c3_pool}

def atomic_json_write(path, payload):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = f"{path}.tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)
        f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)

def load_json(path):
    with open(path, encoding="utf-8") as f: return json.load(f)

def active_emotions(lang):
    return [e for e in EMOTION_ORDER if e not in ABSENT_EMOTIONS.get(lang, set())]

def labels_to_matrix(frame):
    matrix = np.zeros((len(frame), len(EMOTION_ORDER)), dtype=int)
    for idx, e in enumerate(EMOTION_ORDER):
        if e in frame.columns:
            matrix[:, idx] = frame[e].fillna(0).astype(int).to_numpy()
    return matrix

def score_predictions(y_true, y_pred, language):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    raw = {e: float(f1_score(y_true[:,i], y_pred[:,i], zero_division=0))
           for i, e in enumerate(EMOTION_ORDER)}
    result = {e: round(v, 4) for e, v in raw.items()}
    result["macro_f1_raw"] = float(np.mean([raw[e] for e in active_emotions(language)]))
    result["macro_f1"]     = round(result["macro_f1_raw"], 4)
    return result

def pred_path(split, lang, condition):
    return f"{TEST_PRED_DIR}/{split}_{lang}_srng_{condition}.json"

print(f"Phase 4 SERENGETI validation output: {P4_DIR}")
print(f"Target languages: {RUN_LANGS}")

# ── Published benchmark reference ───────────
BENCHMARK_A = {
    "eng": 0.823, "hin": 0.926, "rus": 0.901, "hau": 0.751, "kin": 0.657,
    "sun": 0.550, "yor": 0.461, "vmw": 0.325, "pcm": 0.674,
}
BENCHMARK_C = {
    "eng": 0.797, "hin": 0.919, "rus": 0.906, "hau": 0.709, "kin": 0.519,
    "sun": 0.467, "yor": 0.359, "vmw": 0.210, "pcm": 0.674,
}
BENCHMARK_STATUS = (
    "Descriptive context only: Phase 4 is an internal CAST ablation, "
    "not an official Track A/C submission"
)

Phase 4 SERENGETI validation output: /content/drive/MyDrive/EmotionDetection/CAST_checkpoints/Phase4
Target languages: ['hau', 'kin', 'yor', 'vmw', 'pcm']


In [ ]:
# Load source-language training data for few-shot pool construction
def load_split(language, split):
    cache_path = f"{CACHE_DIR}/{language}_{split}.parquet"
    if os.path.exists(cache_path):
        frame = pd.read_parquet(cache_path)
    else:
        dataset   = load_dataset("brighter-dataset/BRIGHTER-emotion-categories", language)
        aliases   = {"validation":["validation","dev"], "train":["train"], "test":["test"]}
        split_key = next((k for k in aliases[split] if k in dataset), None)
        if split_key is None: raise KeyError(f"{language}: no {split} split")
        frame = dataset[split_key].to_pandas()
        for e in EMOTION_ORDER:
            if e not in frame.columns: frame[e] = 0
        if "text" not in frame.columns: raise ValueError(f"{language}/{split}: text missing")
        os.makedirs(os.path.dirname(cache_path), exist_ok=True)
        frame.to_parquet(cache_path, index=False)
    for e in EMOTION_ORDER:
        if e not in frame.columns: frame[e] = 0
    return frame.reset_index(drop=True)

# Collect all source language training sets needed
required_sources = set()
for target in RUN_LANGS:
    for cfg in CONFIG_ORDER:
        required_sources.update(POOL_FUNCTIONS[cfg](target))

SOURCE_TRAIN_DATA = {}
for src in sorted(required_sources):
    try:
        SOURCE_TRAIN_DATA[src] = load_split(src, "train")
    except Exception as exc:
        print(f"WARNING: {src} training data unavailable: {exc}")

print(f"Loaded {len(SOURCE_TRAIN_DATA)} source-language training sets.")


Loaded 14 source-language training sets.


In [ ]:
CULTURAL_CONTENT = {'eng': {'p1': """English is a low-context, individualist language where emotions are expressed
               directly and individually. The dominant cultural script values emotional
               self-control - managing displays is considered a sign of maturity. Positive
               emotions are expected openly in public; smiling toward strangers is a default
               social norm. Direct verbal labelling is the primary channel - English speakers
               name what they feel rather than conveying it indirectly. Individual emotional
               autonomy is prioritised over communal regulation. The Anglo cultural ideal
               emphasises managing emotion through rational self-direction rather than yielding
               to it or suppressing it entirely.""",
         'p2': """Joy: Publicly expressed and socially expected. Smiling toward strangers is
               normative. Happiness framed as individual achievement and personal wellbeing
               rather than communal experience. Sadness: Expressed directly through verbal
               statement. Prolonged public grief is socially uncomfortable - composure is
               expected relatively quickly. Sadness framed as temporary and manageable. Anger:
               Direct verbal expression culturally acceptable, particularly as assertiveness.
               Anger at injustice is legitimate. Aggressive physical displays are socially
               sanctioned against. Fear: Acknowledged verbally and directly. Self-disclosure of
               vulnerability is acceptable in appropriate contexts - less stigmatised than in
               high-context cultures. Surprise: Expressed openly with verbal exclamations. More
               restrained in formal contexts, more exuberant in casual speech. Disgust: Primarily
               sensory and physical rather than moral - triggered by contamination, bodily
               functions, and violations of physical purity. Maps to direct sensory repulsion
               more than moral transgression."""},
 'hin': {'p1': 'Hindi is a high-context, collectivist language where emotions are shaped by family '
               'honour, social hierarchy, and communal harmony. Indirect expression is the norm - '
               'conveyed through implication, silence, and bodily metaphors rather than direct '
               'statement. Izzat (honour) functions as a collective family asset; emotions '
               'threatening reputation are suppressed. Lajjā (modesty/shame) is a cultural virtue '
               'acting as public restraint, especially with elders. Collectivist display rules '
               'prioritise social harmony - negative emotions are masked to avoid disrupting group '
               'relations. Low-arousal states like contentment and peace are more valued than '
               'exuberant positivity. Emotions are expressed in Hindi-English code-mixed registers '
               'online.',
         'p2': 'Joy (khushi/sukh): Sukh is deep internal contentment without outward display. '
               'Happiness conceptualised through sweetness, illumination, and moral goodness. Tied '
               'to communal festivals not individual pleasure. Sadness (dukh/gham): Expressed '
               'through dukh (suffering), gham (grief), udaas (melancholy). Absorbed silently to '
               'protect family harmony. Anger (gussa/krodh): Gussa suppressed toward elders, '
               'acceptable toward subordinates. Krodh is intense destructive anger. Izzat '
               'violation is primary elicitor. Somaticised through reddening face and gnashing of '
               'teeth. Conceptualised as fluid, storm, or wild animal. Fear (darr/ghabrahat): '
               'Direct acknowledgement rare. Triggers: izzat violation and social judgement. '
               'Burden expressed through dil par patthar rakhna - placing a stone on the heart. '
               'Surprise (ashcharya): Less exuberant than English. Terms: ashcharya, adbhut. '
               'Exclamation arre marks sudden shift. Disgust (ghrina): No direct English '
               'equivalent. Maps to moral violation and ritual impurity rooted in Hindu '
               'purity-pollution norms, not sensory repulsion.'},
 'rus': {'p1': 'Russian emotional expression is heavily characterized by a cultural norm of '
               'stoicism in public spaces. Russians are culturally discouraged from displaying '
               'strong positive or negative emotions publicly, and actions like smiling at '
               'strangers are often considered insincere or inappropriate. In stark contrast to '
               'this public restraint, there is a deep cultural allowance for intense, uninhibited '
               'emotional expression within private, trusted relationships. The culture places '
               'significant value on enduring suffering, endurance, and melancholy, viewing these '
               'states as possessing an almost virtuous quality. This is encapsulated in the '
               'untranslatable concept of toska, which describes a profound longing or melancholy. '
               'While daily verbal expression may be restrained, the Russian literary and poetic '
               'tradition serves as a vibrant channel for emotions that are otherwise suppressed. '
               'Consequently, many complex emotional concepts exist primarily within this elevated '
               'literary register rather than in everyday speech. Furthermore, emotions are '
               'closely tied to physical embodiment, as linguistic collocations heavily treat the '
               'body as an organ of emotional expression. However, it is important to note that '
               'online emotional expression on platforms like VKontakte or Telegram has developed '
               'distinct, more disinhibited norms that diverge from traditional offline stoicism.',
         'p2': 'Joy is known as radost, but it is typically expressed privately or exclusively '
               'within close, trusted relationships. Communal celebration is reserved for specific '
               'occasions, as there is a widespread cultural suspicion of excessive or unearned '
               "public positivity. Sadness is conceptualised as grust' or the deeper toska, which "
               'represents a profound melancholy lacking a direct object. Enduring this sadness '
               'quietly is highly valued, reflecting the cultural virtue of suffering and '
               'resilience. Anger, termed gnev, is subject to strict public suppression but can '
               'reach extreme intensity in private settings. It becomes acceptable to display '
               'publicly only under specific conditions where the grievance is widely recognised '
               'as justified. Fear is termed strakh, and it is often expressed indirectly through '
               'dark humour and deflection rather than through direct, vulnerable acknowledgment. '
               "It may also be channelled through physical descriptions of the body's reaction "
               'rather than naming the emotion itself. Surprise is generally less effusive than '
               'its English equivalents. The linguistic markers used to indicate shock or surprise '
               'convey different levels of intensity and are applied more conservatively. Disgust '
               'is frequently framed in moral and aesthetic terms rather than purely physical '
               'revulsion. It is often expressed through an elevated literary or ironic register '
               'rather than via blunt, direct statements.'},
 'hau': {'p1': 'Hausa emotional expression is heavily shaped by Islamic cultural influence and the '
               'core concept of kunya, which dictates a profound sense of shame and reserve in '
               'social interactions. This cultural framework requires emotional restraint, '
               'particularly in the presence of elders or those owed respect, ensuring that '
               'communal harmony is prioritized over individual outbursts. Another guiding '
               'cultural value is haƙuri, which prescribes patience, composure, and acceptance '
               'under distress or hardship. Consequently, Hausa speakers express emotions through '
               'a rich system of conventionalised indirect patterns rather than direct verbal '
               'statements. These indirect channels include deliberate silence, specific hand '
               'gestures, paralinguistic sounds, and the strategic use of proverbs to convey '
               'feelings without violating social norms. Furthermore, emotional language '
               'frequently relies on bodily metaphors, particularly using the word ciki, meaning '
               'stomach or heart area, to locate feeling within the physical body. Emotional '
               'display is also strictly moderated by gender, as what is considered acceptable '
               'expression differs significantly for men and women within the patriarchal '
               'structure. In addition, the deeply institutionalised belief in aljannu, or '
               'spirits, and tsafi, or witchcraft, functions as a powerful tool for social control '
               'and emotional regulation. Women, who face restrictive patriarchal structures, '
               'sometimes channel suppressed anger through indigenous prose fiction and '
               'metaphorical bodily expressions to safely voice dissent.',
         'p2': 'Joy is articulated as farin ciki, which translates literally to white stomach, '
               'framing happiness as a positive bodily state. Rather than through individual '
               'displays of pleasure, joy is expressed via communal celebrations, incorporating '
               'traditional songs and proverbs. Sadness is conceptualised as baƙin ciki, '
               'translating to black stomach. When dealing with grief, condolence greetings known '
               "as gaisuwar ta'aziya rely heavily on Islamic prayers and euphemisms, strictly "
               'avoiding direct verbal statements about death. Anger features multiple verbs of '
               'varying intensity, such as fusata, tunzura, harzua, and hasala, but direct '
               'expression is suppressed by the rules of kunya. Instead, anger is signalled '
               'through visual bodily cues like zare ido, meaning to pull the eye, or contempt '
               'idioms like sha kunu, meaning to drink gruel. Fear is strongly tied to the '
               'supernatural, particularly anxiety regarding aljannu and tsafi. The pervasive fear '
               'of spirit possession operates as a mechanism of social control, meaning fear is '
               'often expressed through religious or superstitious frameworks. Surprise is '
               'communicated through exclamatory sounds and facial mimicry rather than explicit '
               'verbal statements. This non-verbal approach remains consistent with the broader '
               'cultural norm of indirect emotional expression. Disgust is framed primarily as a '
               'moral and religious violation rather than a physical or sensory repulsion. Taboo '
               'violations that trigger disgust are sanctioned through social stigma and the '
               'threat of supernatural consequences.'},
 'kin': {'p1': 'The emotional landscape in Rwanda is fundamentally shaped by the post-genocide '
               'cultural context, where emotional restraint and silence are deeply sanctioned '
               'responses to trauma and distress. Rwanda possesses a documented culture of '
               'silence, meaning that strong emotions are expected to be processed internally '
               'rather than expressed in public spaces. Emotional management is largely governed '
               'by the concept of agaciro, which translates to dignity and self-worth, placing a '
               'premium on composure. Consequently, collective identity is prioritized over '
               'individual emotional display, making community solidarity the primary vessel for '
               'feelings. Communal rituals and structured environments, such as the gacaca justice '
               'proceedings, provide the sanctioned contexts for collective emotional expression. '
               'Vernacular memory practices and community solidarity create culturally specific '
               'avenues for mourning and processing shared history. Emotions tied directly to the '
               'collective trauma of the genocide do not map cleanly onto Western diagnostic or '
               'emotional categories. Thus, Rwandan emotional expression operates as a highly '
               'regulated system of communal processing and dignified restraint rather than a lack '
               'of feeling.',
         'p2': 'Joy is expressed through communal celebration framed heavily by the concept of '
               'agaciro and collective achievement. Individual displays of joy are expected to be '
               'modest, whereas communal joys during national events, church services, or family '
               'milestones are much more visible and acceptable. Sadness is normatively held '
               'internally as part of the broader culture of silence and dignified restraint. '
               'Public mourning is always communal and highly ritualised, relying on vernacular '
               'mourning practices through community memory rather than direct, individual verbal '
               'grief. Anger is publicly suppressed because overt individual aggression violates '
               'the sanctioned culture of silence and restraint. Instead, collective anger is '
               'processed formally through community mechanisms like the gacaca proceedings, '
               'maintaining a communal rather than an individual frame. Fear is prominently '
               'encapsulated by the term ihahamuka, a Rwanda-specific panic and fear response '
               'rooted in genocide trauma that lacks a Western equivalent. This concept combines '
               'feelings of fear, profound bodily distress, and collective memory into a unified '
               'expression. Surprise relies on restrained interpersonal cues rather than overt '
               'verbal exclamations. Exclamative structures using the question marker mbêga and '
               'manner noun ukūntu are characteristic markers, and the interjection yō signals '
               'sudden amazement. Disgust is expressed through social and moral framing tied to '
               'community reputation. Ishyano, meaning ritual impurity, and kuneena, the '
               'institutionalised avoidance of the morally contaminating, are primary mechanisms. '
               'The verb vugisha encodes the act of disgusting or sickening others.'},
 'sun': {'p1': 'Sundanese emotional expression is deeply rooted in the Austronesian cultural '
               'philosophy of Silih Asah, Silih Asih, and Silih Asuh, meaning mutual learning, '
               'mutual affection, and mutual care. This philosophical framework structures '
               'emotional expression as a fundamentally relational and communal experience rather '
               'than a purely individual one. Additionally, Islamic religious principles strongly '
               'influence emotional norms, demanding significant restraint, particularly regarding '
               'negative emotions like anger. In most interpersonal interactions, maintaining a '
               'flat facial expression serves as the dominant social cue to preserve harmony. '
               'Because overt facial displays are restricted, high vocal intonation and physical '
               'pointing gestures replace direct verbal emotional expression. Sundanese speakers '
               'also frequently utilize local cooperative frameworks like gotong royong to channel '
               'feelings into collective action. Pamali, the concept of taboo, and Islamic norms '
               'jointly regulate public emotional expression. The Lemes speech register cushions '
               'emotionally threatening communication and softens the expression of negative '
               'feelings. Consequently, research indicates that Sundanese speakers exhibit higher '
               'empathy orientations compared to other Indonesian ethnic groups.',
         'p2': 'Joy is expressed through communal celebrations and shared activities like the '
               'botram tradition that actively reflect the philosophy of Silih Asih. Individual '
               'joy is consistently framed in terms of relational harmony and community benefit '
               'rather than isolated personal pleasure. The concept of bodas, meaning white, '
               'encodes purity and happiness, and happiness is defined as virtuous living and '
               'inner calm rather than hedonic pleasure. Sadness is termed sedih or galau, and it '
               'is expressed indirectly through highly restrained body language and specific vocal '
               'intonations. The masking norm of crying in the heart while smiling on the face is '
               'culturally embedded. Grief is also channelled through traditional music, where '
               'madenda tuning evokes a sense of sacred melancholy. Anger is primarily non-verbal '
               'because direct verbal anger is socially discouraged and considered disruptive to '
               'relational harmony. Instead, individuals turn to Islamic coping responses such as '
               'wudhu for ritual washing, istighfar for seeking forgiveness, and the deliberate '
               'practice of patience. Fear is captured by the blended concepts of kawatir and '
               'takut, which merge anxiety and fear into a unified emotional state. Supernatural '
               'and spiritual concerns, heavily influenced by Islamic beliefs, remain prominent '
               'triggers for these fearful expressions. Surprise is uniquely marked by the '
               'exclamatory word meuni, meaning such or how, which frequently collocates with '
               'adjectives and interjections. The intensifier pisan amplifies affect, and the '
               'interjection Duh anchors climactic emotional moments. Disgust is expressed '
               'primarily through moral and religious purity framing, with najis, meaning ritually '
               'unclean, functioning as the primary trigger for jijik, the Sundanese term for '
               'disgust.'},
 'yor': {'p1': 'Yoruba emotional expression is defined by a collectivist community orientation '
               'where feelings are rarely stated directly. Instead, emotions are channelled '
               'through an elaborate system of proverbs known as òwe, as well as through '
               'metaphors, folksongs, and oral poetry. Traditional festivals and communal '
               'gatherings provide the culturally sanctioned spaces for the collective processing '
               'of immense joy and grief. Outside of these communal rituals, individual emotional '
               'display is strictly moderated by powerful social norms regarding face, age, and '
               'respect. Direct confrontation is considered culturally inappropriate, meaning '
               'negative emotions are particularly subject to proverbial indirection. Proverbs '
               'function dynamically as both weapons of social power and diplomatic tools for '
               'conflict resolution. Furthermore, meaning is heavily supplemented by non-verbal '
               'semiotics, including specific hand gestures and facial expressions that carry '
               'exact cultural weight. Therefore, Yoruba emotional communication relies on a '
               'shared, highly contextual understanding of oral literature and bodily metaphor '
               'rather than explicit individual declaration.',
         'p2': 'Joy is expressed collectively through folksongs and traditional festivals that '
               'function as communal soul-menders. It is consistently framed in terms of community '
               'wellbeing, family harmony, and shared prosperity rather than isolated personal '
               'pleasure. Sadness is articulated indirectly through the recitation of proverbs, '
               'traditional dirges, and folksongs. The culture employs a strong somatic '
               'orientation, frequently using bodily organs like the heart metaphorically to '
               'locate and process sorrow. Anger relies heavily on proverbs acting as '
               'face-threatening acts, where powerful sayings function as indirect threats or '
               'insults to avoid discouraged direct confrontation. When described physically, '
               'anger utilizes heat and fire metaphors to illustrate how the emotion violently '
               'overwhelms the body. Fear is powerfully signalled non-verbally through the face of '
               'earnest, a culturally specific facial expression indicating grave seriousness that '
               'is easily misread cross-culturally. Additionally, speakers rely on describing '
               'sudden bodily sensations to locate fear rather than stating the emotion '
               'abstractly. Surprise is communicated via exclamatory interjections and abrupt '
               'shifts in body language. The symbolic placement of an ààlè, using objects like '
               'sand, leaves, or red cloth, conveys non-verbal warning and shock. Disgust is '
               'expressed through proverbs that explicitly invoke moral violation and the breach '
               'of social taboos. It represents a collective community judgement regarding '
               'disrespect for cultural norms rather than an individual sensory reaction.'},
 'vmw': {'p1': 'Emakhuwa emotional expression is profoundly shaped by its matrilineal Bantu social '
               'structure, which dictates distinct gender roles for processing feelings. Women '
               'hold a central role in communal ritual and the public expression of grief, whereas '
               'male gender norms strictly suppress vulnerable emotions like fear and sadness. '
               'Displaying grief loudly and publicly through ritualised mourning is viewed as a '
               'moral obligation to the community rather than a private, individual act. '
               'Conversely, collective joy is channelled through highly structured avenues like '
               'the tufo competitive dance and the Nakhula ancestral dance. Emotional management '
               'is also governed by the concept of ehaya, representing a form of shame tied '
               'intrinsically to communal reputation rather than individual guilt. Furthermore, '
               'collective fear and anger are frequently articulated through culturally specific '
               'idioms of sorcery that possess no direct Western equivalent. It is important to '
               'note that very little academic literature exists regarding Emakhuwa emotional '
               'expression in online contexts, so these offline anthropological norms remain the '
               'primary point of reference.',
         'p2': 'Joy is expressed collectively through events like the tufo competitive dance and '
               'Nakhula ancestral dance during harvests and marriages. Happiness is conceptualised '
               'as a collective communal experience tied to shared celebration and ancestral '
               'ritual. Sadness is expressed through ritualised, public wailing at funerals, which '
               'operates as a strict moral obligation. This loud, collective expression replaces '
               'quiet private grief, while indirect sadness is also processed through traditional '
               'dance and oral tradition. Anger is frequently expressed through sorcery idioms, '
               'utilizing terms like havara for leopard or sorcerer, and ekuluwe for pig to '
               'describe household discord. In broader political contexts, collective resistance '
               'and anger are signalled through phrases like anamalala, meaning it is over. Fear '
               'is heavily tied to sorcery and the mgosyo taboo system regarding hot and cold '
               'states. Anxiety concerning sorcerers among neighbours serves as the dominant '
               'expression of fear, while mgosyo transgressions produce a unique fear-guilt blend '
               'with physical bodily consequences. Surprise possesses very limited direct lexical '
               'expression in the language. Instead, sudden shock is conveyed through specific '
               'interjections and exaggerated body language rather than explicit vocabulary. '
               'Disgust is primarily a moral emotion associated with violations of mgosyo taboos '
               'and subsequent sorcery accusations. Physical or sensory disgust, as it is '
               'understood within Western frameworks, is significantly less prominent.'},
 'pcm': {'p1': 'Nigerian Pidgin, often referred to as Naijà, functions as a vital cross-ethnic '
               "lingua franca that bridges Nigeria's incredibly diverse cultural groups. Because "
               'it deliberately spans various communities, its emotional expression tends to be '
               'much more direct than indigenous languages like Yoruba or Hausa, while still '
               'maintaining its own distinct culturally Nigerian character. Corpus research '
               'indicates that negative sentiment is notably more prevalent in Nigerian language '
               'communities compared to other African language groups. A defining characteristic '
               'of the language is its heavy reliance on unique interjections, which serve as the '
               'most prominent and distinct emotional markers. Furthermore, emotional intensity is '
               'frequently conveyed through the grammatical process of reduplication, where '
               'repeating a word amplifies its feeling. Consistent with wider West African '
               'linguistic patterns, Nigerian Pidgin utilizes bodily sensation constructions that '
               'place emotion directly in the physical body rather than naming it abstractly. '
               'Online, this directness is further amplified, heavily mixing these culturally '
               'specific interjections with global social media norms and emojis.',
         'p2': 'Joy is frequently marked by the exclamatory interjection omo, which signals '
               'intense excitement, admiration, or positive shock. The language relies on communal '
               'celebratory phrasing, strongly favouring these distinctive Pidgin interjections '
               'over their standard English equivalents to express happiness. Sadness is expressed '
               'through bodily sensation constructions, most notably the phrase e pain me, which '
               'locates the sorrow as physical pain. Resigned sorrow or disbelief is captured by '
               'the marker na wa o, while online expressions frequently pair English interjections '
               'alongside sadness emojis. Anger is expressed very directly through phrases like I '
               'vex, meaning I am angry. Intensity is added through reduplication, such as saying '
               'vex vex, and anger frequently co-occurs with expressions of disgust within the '
               'exact same utterance. Fear is conveyed through phrases like I fear am, meaning I '
               'am afraid of it, grounding the feeling in bodily sensations rather than abstract '
               'names. Sharp, repeated pain or fearful distress is also expressed through '
               'sound-symbolic reduplication, such as the term CHUK CHUK. Surprise is marked by '
               'highly distinctive Pidgin interjections like chai and haba. Interestingly, these '
               'shocked expressions tend to occur much more frequently in contexts of negative '
               'surprise than in positive ones. Disgust is widely expressed using the direct '
               'phrase e no good, meaning it is not good. This relies heavily on a dominant moral '
               'framing and frequently clusters together with anger in the same sentence.'}}


def build_cultural_block(language_code, prompt_key):
    content = CULTURAL_CONTENT.get(language_code, {})
    if prompt_key == "p0":
        return ""
    if prompt_key in ("p1", "p2"):
        return content.get(prompt_key, "").strip()
    if prompt_key == "p3":
        combined = "\n\n".join(
            part for part in [
                content.get("p1", "").strip(),
                content.get("p2", "").strip(),
            ]
            if part
        )
        return combined[:3200]
    raise ValueError(f"Unknown prompt key: {prompt_key}")


INSTRUCTION = (
    "Read the text below and identify which emotions it expresses.\n"
    "Available emotions: joy, sadness, anger, fear, surprise, disgust\n"
    "A text may express multiple emotions, one emotion, or none at all.\n\n"
    "Instructions:\n"
    "- Respond ONLY with a comma-separated list of emotion labels from the list above.\n"
    "- Use lowercase exactly as written above.\n"
    "- If no emotion is present, respond with the single word: neutral\n"
    "- Do not add any explanation, punctuation, or extra text."
)


def select_few_shot_examples(target_code, config_key, k=FEW_SHOT_K):
    pool = POOL_FUNCTIONS[config_key](target_code)
    frames = [
        SOURCE_TRAIN_DATA[code]
        for code in pool
        if code in SOURCE_TRAIN_DATA
    ]
    if not frames:
        raise ValueError(
            f"{target_code}/{config_key}: no available source-language training data"
        )

    pool_frame = pd.concat(frames, ignore_index=True)
    examples_by_emotion = {emotion: [] for emotion in EMOTION_ORDER}

    for _, row in pool_frame.iterrows():
        text = str(row.get("text", "")).strip()
        if not text:
            continue
        labels = [
            emotion for emotion in EMOTION_ORDER
            if pd.notna(val := row.get(emotion, 0)) and int(val) == 1
        ]
        if not labels:
            continue
        for emotion in labels:
            examples_by_emotion[emotion].append((text, labels))

    rng = random.Random(RANDOM_SEED + ALL_TARGET_CODES.index(target_code))
    lines = [
        "Here are some example texts in related languages and their emotions:\n"
    ]
    example_index = 1

    for emotion in EMOTION_ORDER:
        candidates = examples_by_emotion[emotion]
        if not candidates:
            continue
        single_label = [
            item for item in candidates if len(item[1]) == 1
        ]
        candidate_pool = single_label if single_label else candidates
        selected = rng.sample(candidate_pool, min(k, len(candidate_pool)))

        for text, labels in selected:
            shortened = text[:120] + "..." if len(text) > 120 else text
            lines.append(
                f'  Example {example_index}: "{shortened}" -> '
                + ", ".join(labels)
            )
            example_index += 1

    if example_index == 1:
        raise ValueError(f"{target_code}/{config_key}: no labelled examples selected")
    lines.append("")
    return "\n".join(lines) + "\n"


In [ ]:
# ── Freeze selections BEFORE loading test data ───────────────────────────────
if not os.path.exists(P2_RESULTS):
    raise FileNotFoundError(f"Run Phase 2 validation first: {P2_RESULTS}")
if not os.path.exists(P3_SRNG_RESULTS):
    raise FileNotFoundError(f"Run Phase 3 SERENGETI validation first: {P3_SRNG_RESULTS}")

p2 = load_json(P2_RESULTS)
p3 = load_json(P3_SRNG_RESULTS)

selection = {
    "selection_split": "validation",
    "prompt_source": P2_RESULTS,
    "config_source": P3_SRNG_RESULTS,
    "languages": {}
}
rows = []
FEWSHOT_BLOCKS = {}

for lang in RUN_LANGS:
    if not all(pr in p2.get(lang, {}) for pr in PROMPT_ORDER):
        raise ValueError(f"Incomplete Phase 2 validation results for {lang}")
    # For SERENGETI, best config from track_a only
    srng_lang = p3.get(lang, {})
    track_a_cfgs = {c: srng_lang[f"track_a_{c}"] for c in CONFIG_ORDER
                    if f"track_a_{c}" in srng_lang
                    and srng_lang[f"track_a_{c}"].get("macro_f1") is not None}
    if not track_a_cfgs:
        raise ValueError(f"No Phase 3 SERENGETI track_a validation results for {lang}")

    bp = max(PROMPT_ORDER,
         key=lambda pr: p2[lang][pr].get("macro_f1_raw", p2[lang][pr]["macro_f1"]))
    bc = max(track_a_cfgs,
         key=lambda c: track_a_cfgs[c].get("macro_f1_raw", track_a_cfgs[c]["macro_f1"]))

    selection["languages"][lang] = {
        "best_prompt": bp,
        "best_config": bc,
        "phase2_val_f1": {pr: p2[lang][pr]["macro_f1"] for pr in PROMPT_ORDER},
        "phase3_srng_track_a_f1": {c: v["macro_f1"] for c, v in track_a_cfgs.items()}
    }
    FEWSHOT_BLOCKS[(lang, bc)] = select_few_shot_examples(lang, bc)
    rows.append({
        "Language code":                                  lang.upper(),
        "Validation-selected prompt":                     bp,
        "Validation-selected transfer configuration":     bc,
        "Selected prompt validation macro-F1":            p2[lang][bp]["macro_f1"],
        "Selected configuration validation macro-F1":     track_a_cfgs[bc]["macro_f1"],
        "Published Track A test best macro-F1 (context only)": BENCHMARK_A[lang],
        "Published Track C test best macro-F1 (context only)": BENCHMARK_C[lang],
        "Benchmark comparison status":                    BENCHMARK_STATUS,
    })

atomic_json_write(SELECTION_PATH, selection)
pd.DataFrame(rows).to_csv(f"{P4_DIR}/phase4_serengeti_validation_selections.csv", index=False)
print("SERENGETI validation selections frozen before loading test data.")
print(pd.DataFrame(rows).drop(columns=["Benchmark comparison status"], errors="ignore").to_string(index=False))


SERENGETI validation selections frozen before loading test data.
Language code Validation-selected prompt Validation-selected transfer configuration  Selected prompt validation macro-F1  Selected configuration validation macro-F1  Published Track A test best macro-F1 (context only)  Published Track C test best macro-F1 (context only)
          HAU                         p2                                         C1                               0.6063                                      0.6974                                                0.751                                                0.709
          KIN                         p1                                         C1                               0.4550                                      0.5946                                                0.657                                                0.519
          YOR                         p1                                         C1                               0.3684  

In [ ]:
# ── Load target test data ONLY AFTER selections are frozen ───────────────────
frozen = load_json(SELECTION_PATH)
assert frozen["selection_split"] == "validation" and set(frozen["languages"]) == set(RUN_LANGS)
TEST_DATA = {lang: load_split(lang, "test") for lang in RUN_LANGS}
for lang, frame in TEST_DATA.items():
    print(f"{lang}: test rows={len(frame)}")
print("Target test data loaded only after SERENGETI selection was frozen.")


hau: test rows=2160
kin: test rows=2462
yor: test rows=3000
vmw: test rows=1554
pcm: test rows=3740
Target test data loaded only after SERENGETI selection was frozen.


In [ ]:
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoModelForMultimodalLM, AutoProcessor

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None
if HF_TOKEN: login(token=HF_TOKEN)

gemma_processor = gemma_model = inner_tokenizer = None
print(f"Loading {GEMMA_MODEL}")
gemma_processor = AutoProcessor.from_pretrained(
    GEMMA_MODEL, token=HF_TOKEN, cache_dir=LOCAL_MODEL_CACHE, trust_remote_code=True)
gemma_model = AutoModelForMultimodalLM.from_pretrained(
    GEMMA_MODEL, token=HF_TOKEN, cache_dir=LOCAL_MODEL_CACHE,
    trust_remote_code=True, dtype="auto", device_map="auto",
    attn_implementation="sdpa").eval()
inner_tokenizer = getattr(gemma_processor, "tokenizer", gemma_processor)
inner_tokenizer.padding_side = "left"
if inner_tokenizer.pad_token_id is None:
    inner_tokenizer.pad_token_id = inner_tokenizer.eos_token_id
GENERATION_CONFIG["pad_token_id"] = inner_tokenizer.pad_token_id
if torch.cuda.is_available():
    print(f"Gemma ready. VRAM: {torch.cuda.memory_allocated()/1e9:.1f}/{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

def apply_chat_template(messages):
    opts = {"tokenize": False, "add_generation_prompt": True, "enable_thinking": False}
    try: return gemma_processor.apply_chat_template(messages, **opts)
    except TypeError:
        opts.pop("enable_thinking"); return gemma_processor.apply_chat_template(messages, **opts)

def build_prompt(lang, text, prompt_key="p0", config_key=None):
    cultural = build_cultural_block(lang, prompt_key)
    if cultural:
        sys_content = "You are a culturally-aware emotion classification system.\n\n" + cultural + "\n\n" + INSTRUCTION
    else:
        sys_content = "You are an emotion classification system.\n" + INSTRUCTION
    if config_key is not None:
        sys_content += "\n\n" + FEWSHOT_BLOCKS[(lang, config_key)].rstrip()
    msgs = [{"role": "user", "content": f"{sys_content}\n\nText: {text}"}]
    return apply_chat_template(msgs) + "Emotions:"

def parse_labels(raw):
    raw = raw.strip().lower()
    if raw in ("","neutral"): return []
    return [l.strip() for l in raw.split(",") if l.strip() in EMOTION_ORDER]

def run_or_load(frame, split, lang, condition, prompt_key, config_key, out_dir):
    out_path = pred_path(split, lang, condition)
    expected_meta = {"split": split, "language": lang, "condition": condition,
                     "prompt": prompt_key, "config": config_key, "emotions": EMOTION_ORDER}
    if os.path.exists(out_path):
        saved = load_json(out_path)
        if all(saved.get(k) == v for k, v in expected_meta.items()):
            y_true = np.asarray(saved["y_true"], dtype=int)
            y_pred  = np.asarray(saved["y_pred"], dtype=int)
            if np.array_equal(y_true, labels_to_matrix(frame)):
                print(f"Reusing {os.path.basename(out_path)}")
                return score_predictions(y_true, y_pred, lang)
    y_true = labels_to_matrix(frame)
    y_pred  = np.zeros_like(y_true)
    texts   = frame["text"].astype(str).tolist()
    progress_path = f"{out_path}.progress.json"
    start = 0
    if os.path.exists(progress_path):
        prog = load_json(progress_path)
        if all(prog.get(k) == v for k, v in expected_meta.items()):
            stored = np.asarray(prog["y_pred"], dtype=int)
            if stored.shape == y_pred.shape:
                y_pred = stored; start = int(prog["processed"])
                print(f"Resuming {lang}/{condition} from {start}/{len(texts)}")
    inner_tokenizer.padding_side = "left"
    for bs in range(start, len(texts), BATCH_SIZE):
        batch_texts = texts[bs: bs + BATCH_SIZE]
        prompts = [build_prompt(lang, t, prompt_key, config_key) for t in batch_texts]
        inputs  = gemma_processor(text=prompts, return_tensors="pt", padding=True,
                                  truncation=True, max_length=2048).to(gemma_model.device)
        with torch.no_grad():
            output_ids = gemma_model.generate(**inputs, **GENERATION_CONFIG)
        pl = inputs["input_ids"].shape[1]
        for li, out in enumerate(output_ids):
            detected = parse_labels(gemma_processor.decode(out[pl:], skip_special_tokens=True))
            row = bs + li
            for ei, e in enumerate(EMOTION_ORDER):
                y_pred[row, ei] = int(e in detected)
        processed = min(bs + BATCH_SIZE, len(texts))
        atomic_json_write(progress_path, {**expected_meta, "processed": processed, "y_pred": y_pred.tolist()})
        if processed % 50 == 0 or processed == len(texts):
            print(f"{lang}/{condition}: {processed}/{len(texts)}")
    atomic_json_write(out_path, {**expected_meta, "y_true": y_true.tolist(), "y_pred": y_pred.tolist()})
    if os.path.exists(progress_path): os.remove(progress_path)
    return score_predictions(y_true, y_pred, lang)


Loading unsloth/gemma-4-31B-it-unsloth-bnb-4bit


processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/18.9k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/5.59k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/22.1k [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1188 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Gemma ready. VRAM: 18.6/42.4 GB


In [ ]:
# ── Four-condition test ablation ───────────────────────────
ablation_results = load_json(RESULTS_PATH) if os.path.exists(RESULTS_PATH) else {}
fewshot_results  = load_json(FEWSHOT_PATH) if os.path.exists(FEWSHOT_PATH) else {}
ablation_rows = []

for lang in RUN_LANGS:
    chosen = frozen["languages"][lang]
    bp, bc = chosen["best_prompt"], chosen["best_config"]
    specs  = {
        "neither":     ("p0", None),
        "prompt_only": (bp,   None),
        "config_only": ("p0", bc),
        "both":        (bp,   bc),
    }
    ablation_results.setdefault(lang, {})
    seen = {}
    print(f"\nTest ablation: {LANGUAGES[lang]['name']} (prompt={bp.upper()}, config={bc})")
    for condition, (prompt, config) in specs.items():
        key = (prompt, config)
        if key in seen:
            src = load_json(pred_path("test", lang, seen[key]))
            src.update(condition=condition, prompt=prompt, config=config)
            atomic_json_write(pred_path("test", lang, condition), src)
            score = score_predictions(src["y_true"], src["y_pred"], lang)
        else:
            score = run_or_load(TEST_DATA[lang], "test", lang, condition, prompt, config, TEST_PRED_DIR)
            seen[key] = condition
        ablation_results[lang][condition] = score
        atomic_json_write(RESULTS_PATH, ablation_results)
        if condition == "config_only":
            fewshot_results[lang] = score
            atomic_json_write(FEWSHOT_PATH, fewshot_results)
        ablation_rows.append({
            "Language code":                                  lang.upper(),
            "Ablation condition":                             condition,
            "Validation-selected prompt":                     bp,
            "Validation-selected transfer configuration":     bc,
            "Macro-F1":                                       score["macro_f1"],
            "Published Track A test best macro-F1 (context only)": BENCHMARK_A[lang],
            "Published Track C test best macro-F1 (context only)": BENCHMARK_C[lang],
            "Benchmark comparison status":                    BENCHMARK_STATUS,
        })
        print(f"  {condition:<14}: {score['macro_f1']:.4f}")

pd.DataFrame(ablation_rows).to_csv(
    f"{P4_DIR}/phase4_serengeti_full_cast_validation_table.csv", index=False)



Test ablation: Hausa (prompt=P2, config=C1)
Reusing test_hau_srng_neither.json
  neither       : 0.5491
Reusing test_hau_srng_prompt_only.json
  prompt_only   : 0.5790
Reusing test_hau_srng_config_only.json
  config_only   : 0.5317
Reusing test_hau_srng_both.json
  both          : 0.5741

Test ablation: Kinyarwanda (prompt=P1, config=C1)
Reusing test_kin_srng_neither.json
  neither       : 0.4223
Reusing test_kin_srng_prompt_only.json
  prompt_only   : 0.4417
Reusing test_kin_srng_config_only.json
  config_only   : 0.4093
Reusing test_kin_srng_both.json
  both          : 0.4374

Test ablation: Yoruba (prompt=P1, config=C1)
Reusing test_yor_srng_neither.json
  neither       : 0.3261
Reusing test_yor_srng_prompt_only.json
  prompt_only   : 0.3202
Reusing test_yor_srng_config_only.json
  config_only   : 0.3143
Reusing test_yor_srng_both.json
  both          : 0.3093

Test ablation: Emakhuwa (prompt=P3, config=C2)
Reusing test_vmw_srng_neither.json
  neither       : 0.0709
Reusing test_vm

In [ ]:
_pw_path  = f"{P4_DIR}/phase4_serengeti_ablation_validation_significance.csv"
_int_path = f"{P4_DIR}/phase4_serengeti_interaction_validation_significance.csv"

if os.path.exists(_pw_path) and os.path.exists(_int_path):
    pw_df  = pd.read_csv(_pw_path)
    int_df = pd.read_csv(_int_path)
    print("Loaded cached bootstrap results (skipping recomputation)")
else:
    def macro_f1_arr(y_true, y_pred, lang):
        keep = [i for i, e in enumerate(EMOTION_ORDER) if e in active_emotions(lang)]
        return float(f1_score(y_true[:,keep], y_pred[:,keep], average="macro", zero_division=0))

    def paired_bootstrap(y_true, pa, pb, lang, n_boot=N_BOOT, seed=RANDOM_SEED):
        rng = np.random.default_rng(seed)
        n   = len(y_true)
        obs = macro_f1_arr(y_true, pa, lang) - macro_f1_arr(y_true, pb, lang)
        draws = np.empty(n_boot)
        for i in range(n_boot):
            idx = rng.integers(0, n, n)
            draws[i] = macro_f1_arr(y_true[idx], pa[idx], lang) - macro_f1_arr(y_true[idx], pb[idx], lang)
        lo, hi = np.percentile(draws, [2.5, 97.5])
        p = float(min(1.0, 2*min(float(np.mean(draws<=0)), float(np.mean(draws>=0)))))
        return obs, lo, hi, p

    def interaction(y_true, neither, prompt_only, config_only, both, lang):
        def mf1(yp): return macro_f1_arr(y_true, yp, lang)
        return mf1(both) - mf1(prompt_only) - mf1(config_only) + mf1(neither)

    pairwise_rows, interaction_rows = [], []
    PAIRS = [("prompt_only vs neither","prompt_only","neither"),
             ("config_only vs neither","config_only","neither"),
             ("both vs neither","both","neither"),
             ("both vs prompt_only","both","prompt_only"),
             ("both vs config_only","both","config_only")]

    for lang in RUN_LANGS:
        preds, ref_y = {}, None
        for cond in ["neither","prompt_only","config_only","both"]:
            payload = load_json(pred_path("test", lang, cond))
            y_true  = np.asarray(payload["y_true"], dtype=int)
            if ref_y is None: ref_y = y_true
            elif not np.array_equal(ref_y, y_true):
                raise ValueError(f"{lang}: gold-label mismatch for {cond}")
            preds[cond] = np.asarray(payload["y_pred"], dtype=int)

        for label, ca, cb in PAIRS:
            seed = RANDOM_SEED + RUN_LANGS.index(lang)
            obs, lo, hi, p = paired_bootstrap(ref_y, preds[ca], preds[cb], lang, seed=seed)
            pairwise_rows.append({
                "Language code":                                  lang.upper(),
                "Language":                                       LANGUAGES[lang]["name"],
                "Validation-selected prompt":                     frozen["languages"][lang]["best_prompt"],
                "Comparison":                                     label,
                "First condition":                                ca,
                "Second condition":                               cb,
                "First-condition macro-F1":                       round(macro_f1_arr(ref_y, preds[ca], lang), 4),
                "Second-condition macro-F1":                      round(macro_f1_arr(ref_y, preds[cb], lang), 4),
                "Macro-F1 difference (first minus second)":       round(obs, 4),
                "95% confidence interval lower bound":            round(lo, 4),
                "95% confidence interval upper bound":            round(hi, 4),
                "Raw p-value":                                    p,
                "Higher-scoring condition":                       ca if obs > 0 else cb,
            })

        rng_i = np.random.default_rng(RANDOM_SEED + RUN_LANGS.index(lang))
        obs_i = interaction(ref_y, preds["neither"], preds["prompt_only"], preds["config_only"], preds["both"], lang)
        draws_i = np.empty(N_BOOT)
        n = len(ref_y)
        for i in range(N_BOOT):
            idx = rng_i.integers(0, n, n)
            draws_i[i] = interaction(ref_y[idx], preds["neither"][idx], preds["prompt_only"][idx],
                                     preds["config_only"][idx], preds["both"][idx], lang)
        lo_i, hi_i = np.percentile(draws_i, [2.5, 97.5])
        p_i = float(min(1.0, 2*min(float(np.mean(draws_i<=0)), float(np.mean(draws_i>=0)))))
        direction = "sub-additive" if obs_i < 0 else ("synergistic" if obs_i > 0 else "exactly additive")
        interaction_rows.append({
            "Language code":                                                  lang.upper(),
            "Language":                                                       LANGUAGES[lang]["name"],
            "Validation-selected prompt":                                     frozen["languages"][lang]["best_prompt"],
            "Interaction effect: both - prompt-only - config-only + neither": round(obs_i, 4),
            "95% confidence interval lower bound":                            round(lo_i, 4),
            "95% confidence interval upper bound":                            round(hi_i, 4),
            "Raw p-value":                                                    p_i,
            "Interaction direction":                                          direction,
        })

    pw_df = pd.DataFrame(pairwise_rows)
    reject, adjusted, _, _ = multipletests(pw_df["Raw p-value"].to_numpy(), method="holm")
    pw_df["Holm-adjusted p-value"] = adjusted
    pw_df["Significant after Holm correction (alpha=0.05)"] = reject
    pw_df["Evaluation split"] = "Test"
    pw_df["Statistical test"] = "Paired bootstrap; Holm correction applied"
    pw_df["Analysis scope"] = "Internal Phase 4 CAST ablation"
    _pw_cols = [
        "Language code", "Language", "Validation-selected prompt", "Comparison",
        "First condition", "Second condition",
        "First-condition macro-F1", "Second-condition macro-F1",
        "Macro-F1 difference (first minus second)",
        "95% confidence interval lower bound", "95% confidence interval upper bound",
        "Raw p-value", "Higher-scoring condition",
        "Holm-adjusted p-value", "Significant after Holm correction (alpha=0.05)",
        "Evaluation split", "Statistical test", "Analysis scope",
    ]
    pw_df = pw_df[_pw_cols]
    pw_df.to_csv(_pw_path, index=False)

    int_df = pd.DataFrame(interaction_rows)
    reject2, adjusted2, _, _ = multipletests(int_df["Raw p-value"].to_numpy(), method="holm")
    int_df["Holm-adjusted p-value"] = adjusted2
    int_df["Significant after Holm correction (alpha=0.05)"] = reject2
    int_df["Evaluation split"] = "Test"
    int_df["Statistical test"] = "Paired bootstrap; Holm correction applied"
    int_df["Analysis scope"] = "Internal Phase 4 CAST ablation"
    _int_cols = [
        "Language code", "Language", "Validation-selected prompt",
        "Interaction effect: both - prompt-only - config-only + neither",
        "95% confidence interval lower bound", "95% confidence interval upper bound",
        "Raw p-value", "Interaction direction",
        "Holm-adjusted p-value", "Significant after Holm correction (alpha=0.05)",
        "Evaluation split", "Statistical test", "Analysis scope",
    ]
    int_df = int_df[_int_cols]
    int_df.to_csv(_int_path, index=False)

print("Pairwise and interaction bootstrap saved")
print(int_df[["Language code","Interaction effect: both - prompt-only - config-only + neither","95% confidence interval lower bound","95% confidence interval upper bound","Holm-adjusted p-value","Interaction direction"]].to_string(index=False))

In [ ]:
_srng_pw_path  = f"{P4_DIR}/phase4_serengeti_ablation_validation_significance.csv"
_srng_int_path = f"{P4_DIR}/phase4_serengeti_interaction_validation_significance.csv"

# Pairwise
srng_pw = pd.read_csv(_srng_pw_path)
srng_pw = srng_pw.rename(columns={
    "lang":          "Language code",
    "language":      "Language",
    "best_prompt":   "Validation-selected prompt",
    "comparison":    "Comparison",
    "condition_a":   "First condition",
    "condition_b":   "Second condition",
    "score_a":       "First-condition macro-F1",
    "score_b":       "Second-condition macro-F1",
    "delta":         "Macro-F1 difference (first minus second)",
    "ci_low":        "95% confidence interval lower bound",
    "ci_high":       "95% confidence interval upper bound",
    "p":             "Raw p-value",
    "winner":        "Higher-scoring condition",
    "adjusted_p":    "Holm-adjusted p-value",
    "sig_adjusted":  "Significant after Holm correction (alpha=0.05)",
})
if "Language code" in srng_pw.columns:
    srng_pw["Language code"] = srng_pw["Language code"].str.upper()
if "Evaluation split" not in srng_pw.columns:
    srng_pw["Evaluation split"] = "Test"
    srng_pw["Statistical test"]  = "Paired bootstrap; Holm correction applied"
    srng_pw["Analysis scope"]    = "Internal Phase 4 CAST ablation"
srng_pw.to_csv(_srng_pw_path, index=False)
print(f"SERENGETI pairwise renamed: {list(srng_pw.columns)}")

# Interaction
srng_int = pd.read_csv(_srng_int_path)
srng_int = srng_int.rename(columns={
    "lang":          "Language code",
    "language":      "Language",
    "best_prompt":   "Validation-selected prompt",
    "interaction":   "Interaction effect: both - prompt-only - config-only + neither",
    "ci_low":        "95% confidence interval lower bound",
    "ci_high":       "95% confidence interval upper bound",
    "p":             "Raw p-value",
    "direction":     "Interaction direction",
    "adjusted_p":    "Holm-adjusted p-value",
    "sig_adjusted":  "Significant after Holm correction (alpha=0.05)",
})
if "Language code" in srng_int.columns:
    srng_int["Language code"] = srng_int["Language code"].str.upper()
if "Evaluation split" not in srng_int.columns:
    srng_int["Evaluation split"] = "Test"
    srng_int["Statistical test"]  = "Paired bootstrap; Holm correction applied"
    srng_int["Analysis scope"]    = "Internal Phase 4 CAST ablation"
srng_int.to_csv(_srng_int_path, index=False)
print(f"✓ SERENGETI interaction renamed: {list(srng_int.columns)}")

✓ SERENGETI pairwise renamed: ['Language code', 'Language', 'Validation-selected prompt', 'Comparison', 'First condition', 'Second condition', 'First-condition macro-F1', 'Second-condition macro-F1', 'Macro-F1 difference (first minus second)', '95% confidence interval lower bound', '95% confidence interval upper bound', 'Raw p-value', 'Higher-scoring condition', 'Holm-adjusted p-value', 'Significant after Holm correction (alpha=0.05)', 'Evaluation split', 'Statistical test', 'Analysis scope']
✓ SERENGETI interaction renamed: ['Language code', 'Language', 'Validation-selected prompt', 'Interaction effect: both - prompt-only - config-only + neither', '95% confidence interval lower bound', '95% confidence interval upper bound', 'Raw p-value', 'Interaction direction', 'Holm-adjusted p-value', 'Significant after Holm correction (alpha=0.05)', 'Evaluation split', 'Statistical test', 'Analysis scope']


In [ ]:
# Integrity check
required = [SELECTION_PATH, RESULTS_PATH, FEWSHOT_PATH,
            f"{P4_DIR}/phase4_serengeti_full_cast_validation_table.csv",
            f"{P4_DIR}/phase4_serengeti_ablation_validation_significance.csv",
            f"{P4_DIR}/phase4_serengeti_interaction_validation_significance.csv"]
missing = [p for p in required if not os.path.exists(p)]
if missing: raise FileNotFoundError("\n".join(missing))
for lang in RUN_LANGS:
    for cond in ("neither","prompt_only","config_only","both"):
        p = pred_path("test", lang, cond)
        if not os.path.exists(p): raise FileNotFoundError(p)
print(f"Phase 4 SERENGETI validation complete - {len(RUN_LANGS)} languages, {P4_DIR}")



✓ Phase 4 SERENGETI validation complete — 5 languages, /content/drive/MyDrive/EmotionDetection/CAST_checkpoints/Phase4
